# BGE reranking: BM25 vs Dense vs Hybrid at chunk level

Evaluates 113 answerable QA using `gold_id` chunk IDs. Each strategy contributes its own fixed top-50 candidate pool; BGE only changes candidate order. Unanswerable QA is evaluated separately by the RAG abstention gate, not retrieval metrics.

In [ ]:
from pathlib import Path
import csv
import json
import time

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    raise RuntimeError('Run notebook from repository root.')

# Output created by src/RAG/retrieval/retrieval_colab_incremental_qa.ipynb.
REPORT_PATH = REPO_ROOT / 'reports/retrieval_eval/token_gold_chunk/per_query.csv'
CHUNKS_PATH = REPO_ROOT / 'data/chunking/output/vieonline_news_chunks_token.jsonl'
INPUT_DIR = REPO_ROOT / 'src/RAG/reranker/input/token_gold_chunk'
OUTPUT_DIR = REPO_ROOT / 'data/reranker/output/token_gold_chunk_bge'

MODEL = 'BAAI/bge-reranker-v2-m3'
CANDIDATE_K = 50
RERANK_TOP_K = 10  # Keep enough ranks for @1/@5/@10 comparison.
BATCH_SIZE = 8      # Lower to 4 if GPU runs out of memory.
MAX_LENGTH = 512    # Match serving RAG reranker configuration.
STRATEGIES = ('bm25', 'dense', 'hybrid')
if not REPORT_PATH.exists():
    raise FileNotFoundError(f'Run retrieval notebook first: {REPORT_PATH}')
if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f'Missing token chunks: {CHUNKS_PATH}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Create one BGE input per retrieval strategy

Adds chunk text to each `retrieved_chunk_id`. Do not merge strategy pools: each BGE run must rerank its own top-50 list.

In [ ]:
from src.RAG.reranker.prepare_retrieval_report_inputs import build_inputs, write_inputs

inputs = build_inputs(REPORT_PATH, CHUNKS_PATH)
assert set(inputs) == set(STRATEGIES)
query_count = len(inputs['bm25'])
assert query_count > 0
assert all(len(inputs[strategy]) == query_count for strategy in STRATEGIES)
assert all(
    len(row['candidates']) == CANDIDATE_K
    for strategy in STRATEGIES for row in inputs[strategy]
)
write_inputs(inputs, INPUT_DIR)
pd.DataFrame({
    'strategy': STRATEGIES,
    'queries': [len(inputs[s]) for s in STRATEGIES],
    'candidates_per_query': [len(inputs[s][0]['candidates']) for s in STRATEGIES],
})

## 2. Rerank each fixed top-50 pool with BGE

First run downloads `BAAI/bge-reranker-v2-m3`. Each output preserves gold chunk IDs, original rank, and BGE score.

In [ ]:
from src.RAG.reranker.bge_reranker import rerank_rows

reranked_by_strategy = {}
rerank_wall_latency_ms = {}
for strategy in STRATEGIES:
    started = time.perf_counter()
    reranked_rows = rerank_rows(
        inputs[strategy], MODEL, RERANK_TOP_K, BATCH_SIZE, MAX_LENGTH
    )
    rerank_wall_latency_ms[strategy] = 1000 * (time.perf_counter() - started) / len(reranked_rows)
    reranked_by_strategy[strategy] = reranked_rows
    output_path = OUTPUT_DIR / f'{strategy}_bge_top{RERANK_TOP_K}.jsonl'
    with output_path.open('w', encoding='utf-8') as handle:
        for row in reranked_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'{strategy}: {len(reranked_rows)} queries, {rerank_wall_latency_ms[strategy]:.1f} ms/query including model load')

## 3. Evaluate chunk-level retrieval after reranking

A hit requires an exact `chunk_id` match with `gold_id`. This measures evidence retrieval for RAG.

In [ ]:
from src.RAG.retrieval.metrics import metrics_for_ranking

def evaluate_rows(rows):
    scored = []
    for row in rows:
        metrics = metrics_for_ranking(
            set(map(str, row['gold_chunk_ids'])),
            [str(candidate['chunk_id']) for candidate in row['reranked_candidates']],
        )
        scored.append({**metrics, 'is_possible': bool(row['is_possible']), 'rerank_latency_ms': float(row['rerank_latency_ms'])})
    return pd.DataFrame(scored)

summary_rows = []
for strategy in STRATEGIES:
    detail = evaluate_rows(reranked_by_strategy[strategy])
    summary_rows.append({
        'strategy': strategy,
        'split': 'answerable',
        'queries': len(detail),
        **detail.drop(columns=['is_possible', 'rerank_latency_ms']).mean().to_dict(),
        'mean_rerank_latency_ms': detail['rerank_latency_ms'].mean(),
        'p50_rerank_latency_ms': detail['rerank_latency_ms'].quantile(.50),
        'p95_rerank_latency_ms': detail['rerank_latency_ms'].quantile(.95),
        'wall_ms_per_query_including_model_load': rerank_wall_latency_ms[strategy],
    })

summary = pd.DataFrame(summary_rows).sort_values(['split', 'ndcg@10'], ascending=[True, False])
summary.to_csv(OUTPUT_DIR / 'bge_rerank_summary.csv', index=False)
display(summary.round(4))

## 4. Compare before vs after BGE

Compare same metric. BGE can improve rank only when a gold chunk already exists in its top-50 candidate pool.

In [ ]:
before = pd.read_csv(REPORT_PATH)
before = before[before['method'].isin(STRATEGIES)].copy()
before['split'] = 'answerable'
before_summary = before.groupby(['method', 'split'], as_index=False)[
    ['hit@1', 'hit@5', 'hit@10', 'recall@1', 'recall@5', 'recall@10', 'mrr@10', 'ndcg@10']
].mean().rename(columns={'method': 'strategy'})

after_summary = summary.drop(columns=['queries', 'mean_rerank_latency_ms', 'p50_rerank_latency_ms', 'p95_rerank_latency_ms', 'wall_ms_per_query_including_model_load'])
comparison = before_summary.merge(after_summary, on=['strategy', 'split'], suffixes=('_before', '_after'))
for metric in ['hit@1', 'hit@5', 'hit@10', 'mrr@10', 'ndcg@10']:
    comparison[f'{metric}_delta'] = comparison[f'{metric}_after'] - comparison[f'{metric}_before']
comparison.to_csv(OUTPUT_DIR / 'bge_before_after_comparison.csv', index=False)
display(comparison.sort_values(['split', 'strategy']).round(4))

## 5. Export selected contexts for offline RAG

One JSONL row per QA and strategy. `contexts` is already final top-10 BGE order and preserves citation metadata. Runtime `NewsPipeline` still retrieves from Qdrant; this export is for reproducible offline RAG evaluation.

In [ ]:
rag_context_path = OUTPUT_DIR / f'rag_contexts_bge_top{RERANK_TOP_K}.jsonl'
with rag_context_path.open('w', encoding='utf-8') as handle:
    for strategy, rows in reranked_by_strategy.items():
        for row in rows:
            contexts = [
                {
                    key: candidate.get(key)
                    for key in ('chunk_id', 'article_id', 'title', 'category', 'url', 'chunk_index', 'text', 'original_rank', 'rerank_score', 'rank')
                }
                for candidate in row['reranked_candidates']
            ]
            handle.write(json.dumps({
                'qa_id': row['qa_id'], 'strategy': strategy, 'question': row['question'],
                'gold_chunk_ids': row['gold_chunk_ids'], 'contexts': contexts,
            }, ensure_ascii=False) + '\n')
print(f'Wrote RAG contexts: {rag_context_path}')